# Inside Steam - SQL Data Preparation

This notebook prepares the cleaned Steam dataset for loading into the PostgreSQL relational database.

The game-level dataset is separated into normalized dimension and bridge tables for genres, publishers and developers.

In [1]:
import pandas as pd
import json
from pathlib import Path

In [2]:
file_path = "../data/processed/games_cleaned_march2025.csv"

df_sql = pd.read_csv(
    file_path,
    low_memory=False
)

df_sql.shape

(94948, 48)

In [3]:
list_columns = [
    "genres",
    "publishers",
    "developers"
]

for col in list_columns:
    df_sql[col] = df_sql[col].apply(json.loads)

In [4]:
print(type(df_sql["genres"].iloc[0]))
print(type(df_sql["publishers"].iloc[0]))
print(type(df_sql["developers"].iloc[0]))

<class 'list'>
<class 'list'>
<class 'list'>


In [5]:
sql_data_path = Path("../data/sql_ready")
sql_data_path.mkdir(parents=True, exist_ok=True)

In [6]:
games_columns = [
    "appid",
    "name",
    "release_date",
    "release_year",
    "required_age",
    "price",
    "discount",
    "is_free",
    "price_band",
    "windows",
    "mac",
    "linux",
    "metacritic_score",
    "recommendations",
    "positive",
    "negative",
    "estimated_owners",
    "owners_lower",
    "owners_upper",
    "estimated_owners_midpoint",
    "average_playtime_forever",
    "median_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_2weeks",
    "peak_ccu",
    "pct_pos_total",
    "num_reviews_total",
    "pct_pos_recent",
    "num_reviews_recent",
    "recent_reception_gap",
    "has_lifetime_playtime",
    "has_recent_playtime",
    "has_peak_ccu",
    "extreme_playtime",
    "header_image"
]

games_sql = df_sql[games_columns].copy()

In [7]:
nullable_integer_columns = [
    "required_age",
    "metacritic_score",
    "owners_lower",
    "owners_upper",
    "estimated_owners_midpoint",
    "num_reviews_total",
    "num_reviews_recent"
]

for col in nullable_integer_columns:
    games_sql[col] = games_sql[col].astype("Int64")

In [8]:
games_sql.to_csv(
    sql_data_path / "games.csv",
    index=False,
    na_rep=""
)

games_sql.shape

(94948, 35)

In [9]:
genres_sql = (
    df_sql[["genres"]]
    .explode("genres")
    .dropna()
    .drop_duplicates()
    .sort_values("genres")
    .reset_index(drop=True)
)

In [10]:
genres_sql["genre_id"] = genres_sql.index + 1

genres_sql = genres_sql[
    ["genre_id", "genres"]
].rename(
    columns={"genres": "genre_name"}
)

genres_sql.head()

,genre_id,genre_name
0,1,360 Video
1,2,Accounting
2,3,Action
3,4,Adventure
4,5,Animation & Modeling


In [11]:
genres_sql.to_csv(
    sql_data_path / "genres.csv",
    index=False
)

In [12]:
game_genres_sql = (
    df_sql[["appid", "genres"]]
    .explode("genres")
    .dropna(subset=["genres"])
)

In [13]:
game_genres_sql = game_genres_sql.merge(
    genres_sql,
    left_on="genres",
    right_on="genre_name",
    how="left"
)

In [14]:
game_genres_sql = (
    game_genres_sql[
        ["appid", "genre_id"]
    ]
    .drop_duplicates()
)

game_genres_sql.head(10)

,appid,genre_id
0,730,3
1,730,13
2,578080,3
3,578080,4
4,578080,17
5,578080,13
6,570,3
7,570,28
8,570,13
9,271590,3


In [15]:
game_genres_sql.to_csv(
    sql_data_path / "game_genres.csv",
    index=False
)

## Publishers and Developers

Publisher and developer fields are normalized into separate dimension tables and many-to-many bridge tables, following the same relational structure used for genres.

In [16]:
publishers_sql = (
    df_sql[["publishers"]]
    .explode("publishers")
    .dropna()
    .drop_duplicates()
    .sort_values("publishers")
    .reset_index(drop=True)
)

publishers_sql["publisher_id"] = publishers_sql.index + 1

publishers_sql = publishers_sql[
    ["publisher_id", "publishers"]
].rename(
    columns={"publishers": "publisher_name"}
)

publishers_sql.head()

,publisher_id,publisher_name
0,1,
1,2,!ReTigma Studio
2,3,#30A6D-S (Kuma)
3,4,#PragmaBreak
4,5,$mitE


In [17]:
game_publishers_sql = (
    df_sql[["appid", "publishers"]]
    .explode("publishers")
    .dropna(subset=["publishers"])
)

In [18]:
game_publishers_sql = game_publishers_sql.merge(
    publishers_sql,
    left_on="publishers",
    right_on="publisher_name",
    how="left"
)

game_publishers_sql = (
    game_publishers_sql[
        ["appid", "publisher_id"]
    ]
    .drop_duplicates()
)

game_publishers_sql.head()

,appid,publisher_id
0,730,42467
1,578080,20914
2,570,42467
3,271590,33840
4,488824,41798


In [19]:
publishers_sql.to_csv(
    sql_data_path / "publishers.csv",
    index=False
)

game_publishers_sql.to_csv(
    sql_data_path / "game_publishers.csv",
    index=False
)

In [20]:
developers_sql = (
    df_sql[["developers"]]
    .explode("developers")
    .dropna()
    .drop_duplicates()
    .sort_values("developers")
    .reset_index(drop=True)
)

developers_sql["developer_id"] = developers_sql.index + 1

developers_sql = developers_sql[
    ["developer_id", "developers"]
].rename(
    columns={"developers": "developer_name"}
)

developers_sql.head()

,developer_id,developer_name
0,1,!CyberApex (SkagoGames)
1,2,!ReTigma Studio
2,3,"""Nieko"""
3,4,#12
4,5,#30A6D-S (Kuma)


In [21]:
game_developers_sql = (
    df_sql[["appid", "developers"]]
    .explode("developers")
    .dropna(subset=["developers"])
)

game_developers_sql = game_developers_sql.merge(
    developers_sql,
    left_on="developers",
    right_on="developer_name",
    how="left"
)

game_developers_sql = (
    game_developers_sql[
        ["appid", "developer_id"]
    ]
    .drop_duplicates()
)

game_developers_sql.head()

,appid,developer_id
0,730,51382
1,578080,36281
2,570,51382
3,271590,41038
4,488824,50632


In [22]:
developers_sql.to_csv(
    sql_data_path / "developers.csv",
    index=False
)

game_developers_sql.to_csv(
    sql_data_path / "game_developers.csv",
    index=False
)

In [23]:
sql_tables = {
    "games": games_sql,
    "genres": genres_sql,
    "game_genres": game_genres_sql,
    "publishers": publishers_sql,
    "game_publishers": game_publishers_sql,
    "developers": developers_sql,
    "game_developers": game_developers_sql
}

for name, table in sql_tables.items():
    print(f"{name}: {table.shape}")

games: (94948, 35)
genres: (33, 2)
game_genres: (258257, 2)
publishers: (49860, 2)
game_publishers: (92617, 2)
developers: (60119, 2)
game_developers: (98015, 2)


In [24]:
print("Missing genre IDs:", game_genres_sql["genre_id"].isna().sum())
print("Missing publisher IDs:", game_publishers_sql["publisher_id"].isna().sum())
print("Missing developer IDs:", game_developers_sql["developer_id"].isna().sum())

Missing genre IDs: 0
Missing publisher IDs: 0
Missing developer IDs: 0


In [25]:
print("Rows:", len(publishers_sql))
print("Missing names:", publishers_sql["publisher_name"].isna().sum())
print("Duplicate names:", publishers_sql["publisher_name"].duplicated().sum())

print(
    "Null characters:",
    publishers_sql["publisher_name"]
    .astype(str)
    .str.contains("\x00", regex=False)
    .sum()
)

print(
    "Names with line breaks:",
    publishers_sql["publisher_name"]
    .astype(str)
    .str.contains(r"[\r\n]", regex=True)
    .sum()
)

Rows: 49860
Missing names: 0
Duplicate names: 0
Null characters: 0
Names with line breaks: 0


In [26]:
publishers_check = pd.read_csv(
    "../data/sql_ready/publishers.csv"
)

publishers_check.shape

(49860, 2)

In [27]:
publishers_check.head()

,publisher_id,publisher_name
0,1,NaN
1,2,!ReTigma Studio
2,3,#30A6D-S (Kuma)
3,4,#PragmaBreak
4,5,$mitE


In [28]:
publishers_sql = (
    df_sql[["publishers"]]
    .explode("publishers")
    .dropna()
)

# Remove accidental whitespace and empty strings
publishers_sql["publishers"] = (
    publishers_sql["publishers"]
    .astype(str)
    .str.strip()
)

publishers_sql = (
    publishers_sql[
        publishers_sql["publishers"] != ""
    ]
    .drop_duplicates()
    .sort_values("publishers")
    .reset_index(drop=True)
)

publishers_sql["publisher_id"] = publishers_sql.index + 1

publishers_sql = publishers_sql[
    ["publisher_id", "publishers"]
].rename(
    columns={"publishers": "publisher_name"}
)

In [29]:
game_publishers_sql = (
    df_sql[["appid", "publishers"]]
    .explode("publishers")
    .dropna(subset=["publishers"])
)

game_publishers_sql["publishers"] = (
    game_publishers_sql["publishers"]
    .astype(str)
    .str.strip()
)

game_publishers_sql = game_publishers_sql[
    game_publishers_sql["publishers"] != ""
]

game_publishers_sql = game_publishers_sql.merge(
    publishers_sql,
    left_on="publishers",
    right_on="publisher_name",
    how="left"
)

game_publishers_sql = (
    game_publishers_sql[
        ["appid", "publisher_id"]
    ]
    .drop_duplicates()
)

In [30]:
publishers_sql.to_csv(
    sql_data_path / "publishers.csv",
    index=False
)

game_publishers_sql.to_csv(
    sql_data_path / "game_publishers.csv",
    index=False
)

In [31]:
publishers_check = pd.read_csv(
    "../data/sql_ready/publishers.csv"
)

print(publishers_check.shape)
print(publishers_check.isna().sum())
print(publishers_check.head())

(49859, 2)
publisher_id      0
publisher_name    5
dtype: int64
   publisher_id          publisher_name
0             1         !ReTigma Studio
1             2         #30A6D-S (Kuma)
2             3            #PragmaBreak
3             4                   $mitE
4             5  ((no-end-parens Studio


In [32]:
publishers_check = pd.read_csv(
    "../data/sql_ready/publishers.csv",
    keep_default_na=False
)

In [33]:
print(publishers_check.shape)

print(
    "Actually empty publisher names:",
    (publishers_check["publisher_name"].str.strip() == "").sum()
)

(49859, 2)
Actually empty publisher names: 0


In [34]:
publishers_default = pd.read_csv(
    "../data/sql_ready/publishers.csv"
)

problem_indices = publishers_default[
    publishers_default["publisher_name"].isna()
].index

publishers_check.loc[problem_indices]

,publisher_id,publisher_name
26958,26959,N/A
26970,26971,NA
28252,28253,None
47086,47087,n/a
47205,47206,null


In [35]:
invalid_labels = {
    "",
    "na",
    "n/a",
    "none",
    "null"
}

In [36]:
# Values that actually mean "publisher unknown"
invalid_labels = {"", "na", "n/a", "none", "null"}

# ----- Publishers table -----

publishers_sql = (
    df_sql[["publishers"]]
    .explode("publishers")
    .dropna()
)

# Clean publisher names
publishers_sql["publishers"] = (
    publishers_sql["publishers"]
    .astype(str)
    .str.strip()
)

# Remove empty / placeholder publisher names
publishers_sql = publishers_sql[
    ~publishers_sql["publishers"]
    .str.lower()
    .isin(invalid_labels)
]

# Keep one row per publisher
publishers_sql = (
    publishers_sql
    .drop_duplicates()
    .sort_values("publishers")
    .reset_index(drop=True)
)

# Create publisher IDs
publishers_sql["publisher_id"] = publishers_sql.index + 1

publishers_sql = publishers_sql[
    ["publisher_id", "publishers"]
].rename(
    columns={"publishers": "publisher_name"}
)


# ----- Game-Publisher bridge table -----

game_publishers_sql = (
    df_sql[["appid", "publishers"]]
    .explode("publishers")
    .dropna(subset=["publishers"])
)

game_publishers_sql["publishers"] = (
    game_publishers_sql["publishers"]
    .astype(str)
    .str.strip()
)

game_publishers_sql = game_publishers_sql[
    ~game_publishers_sql["publishers"]
    .str.lower()
    .isin(invalid_labels)
]

game_publishers_sql = game_publishers_sql.merge(
    publishers_sql,
    left_on="publishers",
    right_on="publisher_name",
    how="left"
)

game_publishers_sql = (
    game_publishers_sql[
        ["appid", "publisher_id"]
    ]
    .drop_duplicates()
)


# ----- Re-export both corrected CSV files -----

publishers_sql.to_csv(
    sql_data_path / "publishers.csv",
    index=False
)

game_publishers_sql.to_csv(
    sql_data_path / "game_publishers.csv",
    index=False
)

In [37]:
print("Publishers:", publishers_sql.shape)
print("Missing publisher IDs:", game_publishers_sql["publisher_id"].isna().sum())

publishers_sql.head()

Publishers: (49851, 2)
Missing publisher IDs: 0


,publisher_id,publisher_name
0,1,!ReTigma Studio
1,2,#30A6D-S (Kuma)
2,3,#PragmaBreak
3,4,$mitE
4,5,((no-end-parens Studio


In [38]:
# ----- Developers table -----

developers_sql = (
    df_sql[["developers"]]
    .explode("developers")
    .dropna()
)

# Clean developer names
developers_sql["developers"] = (
    developers_sql["developers"]
    .astype(str)
    .str.strip()
)

# Remove empty / placeholder developer names
developers_sql = developers_sql[
    ~developers_sql["developers"]
    .str.lower()
    .isin(invalid_labels)
]

# Keep one row per developer
developers_sql = (
    developers_sql
    .drop_duplicates()
    .sort_values("developers")
    .reset_index(drop=True)
)

# Create IDs
developers_sql["developer_id"] = developers_sql.index + 1

developers_sql = developers_sql[
    ["developer_id", "developers"]
].rename(
    columns={"developers": "developer_name"}
)


# ----- Game-Developer bridge table -----

game_developers_sql = (
    df_sql[["appid", "developers"]]
    .explode("developers")
    .dropna(subset=["developers"])
)

game_developers_sql["developers"] = (
    game_developers_sql["developers"]
    .astype(str)
    .str.strip()
)

game_developers_sql = game_developers_sql[
    ~game_developers_sql["developers"]
    .str.lower()
    .isin(invalid_labels)
]

game_developers_sql = game_developers_sql.merge(
    developers_sql,
    left_on="developers",
    right_on="developer_name",
    how="left"
)

game_developers_sql = (
    game_developers_sql[
        ["appid", "developer_id"]
    ]
    .drop_duplicates()
)


# Re-export corrected CSV files
developers_sql.to_csv(
    sql_data_path / "developers.csv",
    index=False
)

game_developers_sql.to_csv(
    sql_data_path / "game_developers.csv",
    index=False
)

In [39]:
print("Developers:", developers_sql.shape)
print(
    "Missing developer IDs:",
    game_developers_sql["developer_id"].isna().sum()
)

developers_sql.head()

Developers: (60113, 2)
Missing developer IDs: 0


,developer_id,developer_name
0,1,!CyberApex (SkagoGames)
1,2,!ReTigma Studio
2,3,"""Nieko"""
3,4,#12
4,5,#30A6D-S (Kuma)


## SQL Data Preparation Summary

The cleaned Steam dataset was transformed into a normalized structure suitable for loading into PostgreSQL.

The original game-level dataset contained several multi-valued fields, including genres, publishers and developers. These fields were separated into dedicated dimension tables and many-to-many bridge tables.

The final relational structure includes:

- `games`: one record per Steam application
- `genres`: unique genre labels
- `game_genres`: relationships between games and genres
- `publishers`: unique publisher labels
- `game_publishers`: relationships between games and publishers
- `developers`: unique developer labels
- `game_developers`: relationships between games and developers

Additional validation was performed before export. Empty and placeholder publisher/developer labels were removed, duplicate labels were standardized, and all bridge-table foreign-key identifiers were checked for missing values.

Seven SQL-ready CSV files were generated and loaded into the PostgreSQL `inside_steam` database.

The relational model was validated through SQL joins, confirming that games can successfully be connected back to their genres and other related dimensions.

The database is now ready for analytical SQL queries, aggregations, CTEs, window functions and reusable analytical views.